In [ ]:
%maven org.knowm.xchart:xchart:3.8.8
// Setup 1/2: fetch the plotting library

In [ ]:
// ==== Setup 2/2: imports and chart helper ====

import org.knowm.xchart.*;
import org.knowm.xchart.style.Styler;
import java.awt.image.BufferedImage;
import java.util.Arrays;

// chart helper - not part of the lesson
BufferedImage plot(String title, String yLabel, double[] xs, String[] names, double[][] ys) {
    XYChart ch = new XYChartBuilder().width(700).height(400)
            .title(title).xAxisTitle("k").yAxisTitle(yLabel).build();
    ch.getStyler().setLegendPosition(Styler.LegendPosition.InsideNW);
    ch.getStyler().setMarkerSize(0);
    for (int i = 0; i < names.length; i++) ch.addSeries(names[i], xs, ys[i]);
    return BitmapEncoder.getBufferedImage(ch);
}

# Lists

Before we start:
- Finish the comparable / comparator parts from last time
- Quiz questions 1 and 2

## Abstraction

- **Abstraction**: describe what something does without describing how it does it.
- **Abstract data type (ADT)**: a description of a data structure the operations it supports, specified **independently of how it is implemented**.
- In Java: the **interface** is the ADT, each **class** implementing it is one way to realize it.

You have used `String` and `ArrayList` before. What do you know about how they store their data? What did you *need* to know?

## The List ADT

An **ordered sequence** of elements. Each element has a position (index) from `0` to `size - 1`.

### Typical operations

| operation | meaning |
|---|---|
| `add(e)` | append `e` at the end |
| `add(pos, e)` | insert `e` at `pos`; elements from `pos` on move one position up |
| `remove(pos)` | remove and return the element at `pos`; elements after it move one position down |
| `get(pos)`, `set(pos, e)` | read / overwrite the element at `pos` |
| `indexOf(e)` | position of the first occurrence of `e`, or `-1` |
| `size()`, `isEmpty()` | |
| `clear()` | remove all elements |

Not all descriptions of list are identical, but they are all mostly equivalent.

## `MyListDouble`

The ADT written as a Java interface, for a list of `double`s. The comments are part of the contract.

In [ ]:
interface MyListDouble {
    // appends e at the end
    void add(double e);

    // inserts e at pos, elements pos..size-1 move up by one
    // valid: 0 <= pos <= size
    void add(int pos, double e);

    // removes and returns the element at pos, elements pos+1..size-1 move down by one
    // valid: 0 <= pos < size
    double remove(int pos);

    // valid: 0 <= pos < size
    double get(int pos);

    // replaces the element at pos, returns the old one
    // valid: 0 <= pos < size
    double set(int pos, double e);

    // position of the first occurrence of e, or -1
    int indexOf(double e);

    int size();
    boolean isEmpty();
    void clear();
}

A small subset of `java.util.List<E>`.

## Two main implementations

- **Array-based**: elements in consecutive cells of an array. *This week.*
- **Linked**: each element in its own node, nodes reference each other. *Next week.*

Same interface, different costs for different operations!

# Array-based list

## Storage

- `data`: an **array** whose length is called **capacity**
- `size`: number of elements in the list

Elements are stored in `data[0..size-1]` with no gaps. `size <= capacity` and cells `data[size..capacity-1]` are unused.

## Appending

```java
public void add(double e) {
    data[size] = e;
    size++;
}
```

- `O(1)`
- What happens when `size == data.length`?

When the array is full:
- **refuse**: throw an exception, fixed capacity like a plain array
- **grow**: allocate a larger array, copy the elements over, continue

## Growing the array

One grow copies all `size` elements: `O(N)`.

In [ ]:
int[] grow(int[] old, int newCapacity) {
    int[] bigger = new int[newCapacity];
    for (int i = 0; i < old.length; i++)
        bigger[i] = old[i];
    return bigger;
}

int[] a = {1, 2, 3};
a = grow(a, 6);
System.out.println(Arrays.toString(a));

Library versions, same `O(N)`:
- `System.arraycopy(src, srcPos, dest, destPos, length)`: copies a block, implemented natively (faster in practice than the loop)
- `Arrays.copyOf(array, newLength)`: allocates the new array and copies into it

In [ ]:
int[] b = {1, 2, 3};

int[] c = new int[6];
System.arraycopy(b, 0, c, 0, b.length);
System.out.println(Arrays.toString(c));

int[] d = Arrays.copyOf(b, 6);
System.out.println(Arrays.toString(d));

### How much larger?

- too large: wasted memory
- too small: we grow again soon, and copy everything again

Denote with `N` the capacity of the array, and with `C` its initial capacity.

Experiment: append `k` elements, count the element copies.

### Additive growth: `N + G`

Capacities: C, C + G, C + 2G, ... (in the experiment, `C = G`)

In [ ]:
// element copies made while appending k elements, starting from N = G, growing N -> N + G
long copiesAdditive(int k, int G) {
    long copies = 0;
    long N = G;
    for (int size = 0; size < k; size++) {
        if (size == N) {                    // full: copy all N elements
            copies += N;
            N = N + G;
        }
    }
    return copies;
}

System.out.printf("%10s %18s %16s %16s%n", "k", "G = 1", "G = 10", "G = 1000");
for (int k = 1_000; k <= 1_000_000; k *= 10)
    System.out.printf("%,10d %,18d %,16d %,16d%n", k,
                      copiesAdditive(k, 1), copiesAdditive(k, 10), copiesAdditive(k, 1000));

`k` grows x10. What happens to the copies?

Grows at sizes $C, C + G, \dots, C + (m-1)G$, all $< k$, so for $k \gg C$ we grow the array $m \approx k/G$ times.

The $j$-th grow copies $C + jG$ elements:

$$C + (C + G) + \dots + (C + (m-1)G) = mC + G\,\frac{m(m-1)}{2} \approx \frac{Ck}{G} + \frac{k^2}{2G}$$

- `k` appends: `k` writes + about `k²/2G` copies, `O(k²)`
- on average over the `k` appends, each one costs `O(k)`, for **any** constants `C` and `G`
- a larger `G` only divides by a constant

### Multiplicative growth: `2N`

Capacities: C, 2C, 4C, 8C, ... (in the experiment, `C = 8`)

In [ ]:
// element copies made while appending k elements, starting from N = 8, growing N -> 2N
long copiesDoubling(int k) {
    long copies = 0;
    long N = 8;
    for (int size = 0; size < k; size++) {
        if (size == N) {                    // full: copy all N elements
            copies += N;
            N = 2 * N;
        }
    }
    return copies;
}

System.out.printf("%10s %16s %12s%n", "k", "copies", "copies / k");
for (int k = 1_000; k <= 1_000_000; k *= 10)
    System.out.printf("%,10d %,16d %12.2f%n", k, copiesDoubling(k), (double) copiesDoubling(k) / k);

Grows at sizes $C, 2C, 4C, \dots, 2^m C$, with $2^m C < k$:

$$C + 2C + 4C + \dots + 2^m C = C\,(2^{m+1} - 1) < 2 \cdot 2^m C < 2k$$

- `k` appends: `k` writes + fewer than `2k` copies, `O(k)`
- per append: `O(1)`

### Side by side

Copies **per append**: the extra work each `add` pays on average.

In [ ]:
int pts = 100;
double[] ks = new double[pts];
double[][] perAppend = new double[3][pts];
for (int i = 0; i < pts; i++) {
    int k = (i + 1) * 500;                  // 500, 1000, ..., 50000
    ks[i] = k;
    perAppend[0][i] = (double) copiesAdditive(k, 100) / k;
    perAppend[1][i] = (double) copiesAdditive(k, 1000) / k;
    perAppend[2][i] = (double) copiesDoubling(k) / k;
}
plot("Copies per append", "copies / k", ks, new String[] { "G = 100", "G = 1000", "2N" }, perAppend)

### Amortized cost

- **Amortized cost**: total cost of a sequence of `M` operations, divided by `M`, for the **worst** such sequence.
- Not an average over random inputs: a guarantee for every sequence.
- Append with multiplicative growth: a single call is `O(N)` in the worst case, **`O(1)` amortized**.
- Append with additive growth: `O(N)` amortized.

Does the factor have to be 2? What about 1.5? 1.01? What changes?

## `MyArrayListDouble`

Insertion and removal at a position: next time.

In [ ]:
class MyArrayListDouble implements MyListDouble {
    private double[] data;
    private int size;

    MyArrayListDouble(int capacity) {
        if (capacity < 1) throw new IllegalArgumentException("capacity must be positive");
        data = new double[capacity];
        size = 0;
    }

    MyArrayListDouble() { this(10); }

    public int size() { return size; }
    public boolean isEmpty() { return size == 0; }
    int capacity() { return data.length; }       // for our demos; java.util.ArrayList does not expose it

    private void checkIndex(int pos) {
        if (pos < 0 || pos >= size)
            throw new IndexOutOfBoundsException("pos " + pos + ", size " + size);
    }

    public double get(int pos) {
        checkIndex(pos);
        return data[pos];
    }

    public double set(int pos, double e) {
        checkIndex(pos);
        double old = data[pos];
        data[pos] = e;
        return old;
    }

    public int indexOf(double e) {
        for (int i = 0; i < size; i++)
            if (data[i] == e) return i;
        return -1;
    }

    private void grow() {
        data = Arrays.copyOf(data, 2 * data.length);
    }

    public void add(double e) {
        if (size == data.length) grow();
        data[size] = e;
        size++;
    }

    public void add(int pos, double e) { throw new UnsupportedOperationException("not yet"); }
    public double remove(int pos)      { throw new UnsupportedOperationException("not yet"); }

    public void clear() {
        size = 0;
    }

    @Override
    public String toString() {
        return Arrays.toString(Arrays.copyOf(data, size));
    }
}

In [ ]:
MyArrayListDouble list = new MyArrayListDouble(2);
double[] values = {1.5, 2.5, 3.5, 4.5, 5.5};
for (double v : values) {
    list.add(v);
    System.out.println(list + "  size=" + list.size() + "  capacity=" + list.capacity());
}
System.out.println(list.get(2) + " " + list.indexOf(4.5) + " " + list.indexOf(9.9));

Capacity is 8. Is `get(5)` valid?

In [ ]:
list.get(5)

## Exercises

### 1. Counting by hand
Start from capacity 1 and append 17 elements. How many copies with x2? With +3?

### 2. Shrinking
After many removals, `capacity` can be much larger than `size`.
- Propose a policy that halves the array.
- Why is "halve when `size < capacity / 2`" a bad choice? Hint: alternate `add` and `remove` right at the threshold.
- What about `size < capacity / 4`?

In [ ]:
// test helpers
import java.util.NoSuchElementException;

void check(boolean ok, String what) {
    System.out.println((ok ? "ok    " : "FAIL  ") + what);
}

// listOf(1, 2, 3): a new list with those elements
MyArrayListDouble listOf(double... values) {
    MyArrayListDouble l = new MyArrayListDouble();
    for (double v : values) l.add(v);
    return l;
}

### 3. `contains`
Write `boolean contains(double e)`. Can you reuse an existing method?

In [ ]:
MyArrayListDouble t = listOf(1.5, 2.5, 3.5);
check(t.contains(2.5), "element in the middle");
check(t.contains(1.5) && t.contains(3.5), "first and last");
check(!t.contains(4.0), "missing element");
check(!new MyArrayListDouble().contains(1.0), "empty list");

### 4. `lastIndexOf`
Write `int lastIndexOf(double e)`: position of the **last** occurrence, or `-1`.

In [ ]:
MyArrayListDouble t = listOf(1, 2, 1, 3, 1, 4);
check(t.lastIndexOf(1) == 4, "several occurrences");
check(t.lastIndexOf(2) == 1, "single occurrence");
check(t.lastIndexOf(4) == 5, "last element");
check(t.lastIndexOf(9) == -1, "missing element");
check(new MyArrayListDouble().lastIndexOf(1) == -1, "empty list");

### 5. `max`
Write `double max()`: the largest element. On an empty list, throw `NoSuchElementException`. Why not return `0`?

In [ ]:
check(listOf(3, 7, 2).max() == 7, "max in the middle");
check(listOf(9, 7, 2).max() == 9, "max first");
check(listOf(3, 7, 12).max() == 12, "max last");
check(listOf(-5, -2, -9).max() == -2, "all negative");
check(listOf(4).max() == 4, "single element");
try {
    new MyArrayListDouble().max();
    check(false, "empty list should throw");
} catch (NoSuchElementException e) {
    check(true, "empty list throws");
}

### 6. Capacity 0
The constructor rejects capacity 0. What would happen with capacity 0 and our `grow()`? Allow capacity 0 in the constructor and fix `grow()`.

In [ ]:
MyArrayListDouble t = new MyArrayListDouble(0);
check(t.isEmpty(), "starts empty");
for (int i = 0; i < 5; i++) t.add(i);
check(t.size() == 5 && t.get(0) == 0 && t.get(4) == 4, "grows from capacity 0");
try {
    new MyArrayListDouble(-1);
    check(false, "negative capacity should throw");
} catch (IllegalArgumentException e) {
    check(true, "negative capacity rejected");
}

### 7. Constructor from an array
Write `MyArrayListDouble(double[] elements)`: a list that starts with a copy of `elements`. What initial capacity would you choose?

In [ ]:
double[] src = {1, 2, 3};
MyArrayListDouble t = new MyArrayListDouble(src);
check(t.size() == 3 && t.get(0) == 1 && t.get(2) == 3, "same elements");
src[0] = 99;
check(t.get(0) == 1, "a copy, not the same array");
t.add(4);
check(t.size() == 4 && t.get(3) == 4, "can still grow");
MyArrayListDouble e = new MyArrayListDouble(new double[0]);
check(e.isEmpty(), "empty array");
e.add(5);
check(e.size() == 1 && e.get(0) == 5, "empty array, then add");